# 00 · Setup do ambiente

**Objetivo:** preparar a estrutura do projeto no Databricks antes de qualquer carga de dados.

Este notebook:

1. carrega a configuração central do projeto (`src/config.py`);
2. cria o catálogo do projeto no Unity Catalog;
3. cria um schema para cada camada da Arquitetura Medalhão (`bronze`, `silver`, `gold`);
4. cria o volume de *landing*, onde os arquivos brutos ficam antes de virarem tabelas;
5. testa se o portal de dados abertos da ANEEL (Agência Nacional de Energia Elétrica) é acessível a partir do workspace, o que define a estratégia de ingestão.

**Pré-requisito:** repositório do GitHub clonado como *Git folder* no workspace.

**Idempotência:** todas as operações usam `IF NOT EXISTS`; o notebook pode ser reexecutado sem efeitos colaterais.

## 1. Configuração central

Todos os notebooks importam nomes de catálogo, schemas e caminhos de um único arquivo, `src/config.py`.
Uma mudança de nome é feita em um só lugar, sem risco de notebooks apontarem para objetos diferentes.

In [ ]:
import os
import sys

# Notebooks live in <repo>/notebooks; the repo root must be on sys.path to import src/
repo_root = os.path.dirname(os.getcwd())
if repo_root not in sys.path:
    sys.path.append(repo_root)

from src.config import (
    ANEEL_API_URL,
    CATALOG,
    LANDING_PATH,
    LANDING_VOLUME,
    SCHEMA_BRONZE,
    SCHEMA_GOLD,
    SCHEMA_SILVER,
    SOURCE_FOLDERS,
)

print(f"Catalog:      {CATALOG}")
print(f"Schemas:      {SCHEMA_BRONZE}, {SCHEMA_SILVER}, {SCHEMA_GOLD}")
print(f"Landing path: {LANDING_PATH}")

## 2. Catálogo do projeto

O Unity Catalog organiza os objetos em três níveis: `catálogo.schema.tabela`.

**Decisão de projeto:** um catálogo único para o MVP, com um schema por camada.
Um catálogo por camada faz mais sentido quando as camadas têm donos ou políticas de acesso diferentes, o que não ocorre em um projeto individual.

Se o workspace não permitir a criação de catálogos, a célula abaixo interrompe a execução e informa a correção.

In [ ]:
existing_catalogs = [row.catalog for row in spark.sql("SHOW CATALOGS").collect()]

if CATALOG in existing_catalogs:
    print(f"Catalog '{CATALOG}' already exists.")
else:
    try:
        spark.sql(f"CREATE CATALOG {CATALOG}")
        print(f"Catalog '{CATALOG}' created.")
    except Exception as error:
        raise RuntimeError(
            f"Could not create catalog '{CATALOG}'. If this workspace does not allow "
            "catalog creation, set CATALOG = 'workspace' in src/config.py and rerun."
        ) from error

spark.sql(
    f"COMMENT ON CATALOG {CATALOG} IS "
    "'MVP Engenharia de Dados: evolução da qualidade das concessionárias de distribuição de energia elétrica (dados abertos ANEEL)'"
)

## 3. Schemas por camada (Arquitetura Medalhão)

| Schema | Conteúdo | Tratamento |
|---|---|---|
| `bronze` | dados como vieram da fonte, com metadados de ingestão | nenhum |
| `silver` | dados limpos e padronizados | tipagem, deduplicação, padronização, filtros de escopo |
| `gold` | modelo dimensional, métricas e rankings | modelagem e agregação |

In [ ]:
layer_comments = {
    SCHEMA_BRONZE: "Camada Bronze: dados brutos exatamente como recebidos da fonte, acrescidos de metadados de ingestão",
    SCHEMA_SILVER: "Camada Silver: dados limpos, tipados, deduplicados e padronizados",
    SCHEMA_GOLD: "Camada Gold: modelo dimensional, métricas de evolução e rankings",
}

for schema, comment in layer_comments.items():
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema}")
    # COMMENT ON is applied separately so that reruns also update the description
    spark.sql(f"COMMENT ON SCHEMA {CATALOG}.{schema} IS '{comment}'")
    print(f"Schema '{CATALOG}.{schema}' ready.")

## 4. Volume de landing

Um *Volume* é um espaço governado pelo Unity Catalog para arquivos não tabulares (CSV, ZIP, Parquet).
Os arquivos baixados da ANEEL ficam no volume `landing`, em uma pasta por fonte, e só depois são lidos para as tabelas Bronze.
Assim o arquivo original é preservado para rastreabilidade.

In [ ]:
volume_name = f"{CATALOG}.{SCHEMA_BRONZE}.{LANDING_VOLUME}"

spark.sql(f"CREATE VOLUME IF NOT EXISTS {volume_name}")
spark.sql(f"COMMENT ON VOLUME {volume_name} IS 'Arquivos brutos das fontes, uma pasta por conjunto de dados'")

for folder, description in SOURCE_FOLDERS.items():
    os.makedirs(f"{LANDING_PATH}/{folder}", exist_ok=True)
    print(f"{folder:<28} {description}")

## 5. Teste de acesso ao portal da ANEEL

No Databricks Free Edition, o acesso de saída à internet é restrito a um conjunto limitado de domínios.
O teste consulta a API do CKAN (Comprehensive Knowledge Archive Network, plataforma que hospeda o portal de dados abertos da ANEEL) e define a estratégia do notebook `01_bronze_ingestion`:

- **acessível:** download automático via API, reprodutível;
- **bloqueado:** download manual e upload para as pastas do volume `landing`.

In [ ]:
import requests

test_dataset = "indicadores-coletivos-de-continuidade-dec-e-fec"

try:
    response = requests.get(f"{ANEEL_API_URL}/package_show", params={"id": test_dataset}, timeout=30)
    payload = response.json()
    aneel_reachable = response.status_code == 200 and payload.get("success", False)
except (requests.exceptions.RequestException, ValueError) as error:
    # ValueError covers non-JSON answers, such as a proxy block page
    aneel_reachable = False
    print(f"Request failed: {type(error).__name__}: {error}")

if aneel_reachable:
    print("ANEEL portal reachable: notebook 01 will download files through the CKAN API.")
    print(f"License of test dataset: {payload['result'].get('license_title')}")
else:
    print("ANEEL portal not reachable: download files locally and upload them to the landing folders.")

## 6. Evidências

As saídas abaixo servem de screenshot para a seção **Carga dos Dados (Etapa 4.2)** do README.

In [ ]:
display(spark.sql(f"SHOW SCHEMAS IN {CATALOG}"))
display(spark.sql(f"DESCRIBE VOLUME {volume_name}"))
display(dbutils.fs.ls(LANDING_PATH))

## Próximo passo

`01_bronze_ingestion`: carga dos arquivos brutos nas tabelas Bronze, com a estratégia definida pelo teste da seção 5.